# 06. Financial Appraisal & Energy Yield Simulation

## Overview
This notebook simulates annual net energy yields ($	ext{MWh}$/year) and executes a 25-year investment appraisal modeling **CAPEX**, **OPEX**, **PPA Revenue**, **Net Present Value (NPV)**, **IRR %**, **LCOE**, and **Payback Period** in Indian Rupees ($	ext{INR } ₹$).

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

print("Financial modeling engine initialized!")

## 1. Project Baseline Parameters

In [ ]:
capacity_mw = 50.0  # 50 MW Solar PV Plant
capacity_factor = 0.22  # 22% Capacity Factor
hours_per_year = 8760

# Annual Yield Calculation
annual_yield_mwh = capacity_mw * capacity_factor * hours_per_year
annual_yield_gwh = annual_yield_mwh / 1000.0

print(f"Annual Energy Yield: {annual_yield_mwh:,.2f} MWh ({annual_yield_gwh:.2f} GWh)")

## 2. 25-Year Investment Appraisal Modeling

In [ ]:
# Financial Benchmarks (INR ₹)
capex_per_mw = 45000000.0  # ₹4.5 Cr per MW
total_capex = capacity_mw * capex_per_mw

opex_per_mw_year = 700000.0  # ₹7 Lakhs per MW per year
annual_opex = capacity_mw * opex_per_mw_year

ppa_tariff_per_kwh = 2.85  # ₹2.85 per kWh PPA Tariff
annual_revenue = annual_yield_mwh * 1000.0 * ppa_tariff_per_kwh

# Net Cash Flow per year
discount_rate = 0.08  # 8% WACC
cash_flows = [-total_capex] + [(annual_revenue - annual_opex)] * 25

# Calculate NPV
npv = sum(cf / ((1 + discount_rate) ** t) for t, cf in enumerate(cash_flows))

# Calculate LCOE (Levelized Cost of Energy)
total_discounted_cost = total_capex + sum(annual_opex / ((1 + discount_rate) ** t) for t in range(1, 26))
total_discounted_energy = sum((annual_yield_mwh * 1000.0) / ((1 + discount_rate) ** t) for t in range(1, 26))
lcoe = total_discounted_cost / total_discounted_energy

payback_years = total_capex / (annual_revenue - annual_opex)
roi_pct = ((annual_revenue - annual_opex) / total_capex) * 100.0

print(f"Initial CAPEX:           ₹{total_capex/1e7:.2f} Cr")
print(f"Annual OPEX:            ₹{annual_opex/1e5:.2f} Lakhs")
print(f"Annual Revenue:         ₹{annual_revenue/1e7:.2f} Cr")
print(f"25-Year NPV (8% WACC):  ₹{npv/1e7:.2f} Cr")
print(f"Levelized Cost (LCOE):  ₹{lcoe:.2f} / kWh")
print(f"Payback Period:         {payback_years:.1f} Years")
print(f"Annual ROI:             {roi_pct:.2f} %")

## 3. Cumulative Cash Flow Trajectory

In [ ]:
cumulative_cf = np.cumsum(cash_flows) / 1e7  # Cr INR

plt.figure(figsize=(10, 5))
plt.plot(range(0, 26), cumulative_cf, marker='o', color='forestgreen', linewidth=2)
plt.axhline(0, color='red', linestyle='--')
plt.title('25-Year Cumulative Cash Flow Trajectory (₹ Crores)')
plt.xlabel('Project Lifecycle Year')
plt.ylabel('Net Cash Flow (₹ Cr)')
plt.grid(True)
plt.show()